In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import binom, false_discovery_control

In [ ]:
peaks = pd.read_excel(
    "../../data/SECMX/41467_2025_56303_MOESM5_ESM.xlsx",
    sheet_name="3.AssemblyStateCompare"
)
lists = {}
for key, col in {
    "all": "gSEC- all identified",
    "both": "gSEC-Both",
    "HCT116": "gSEC-HCT",
    "HEK293T": "gSEC-HEK"
}.items():
    lists[key] = set(peaks[col].dropna().astype(str).str.strip())
covered = lists["both"] | lists["HCT116"] | lists["HEK293T"]

groups = pd.DataFrame({"gene": sorted(lists["all"] | covered)})
in_hct = groups["gene"].isin(lists["HCT116"])
in_hek = groups["gene"].isin(lists["HEK293T"])
groups["group"] = np.where(in_hct | in_hek, "specific", "shared")
groups["specific_in"] = np.select(
    [in_hct & in_hek, in_hct, in_hek],
    ["both lines", "HCT116", "HEK293T"],
    default="",
)
groups["mutual_peak"] = groups["gene"].isin(lists["both"]).astype(int)

In [ ]:
raw = pd.read_excel(
    "../../data/SECMX/41467_2025_56303_MOESM6_ESM.xlsx",
    sheet_name="4. HEK-HCT_gUFprot_phUFpep"
)
raw = raw[raw["ptm"] == "global"].copy()
raw["gene"] = raw["gene"].astype(str).str.strip()
raw["context"] = raw["condition"].map({
    "HCT": "HCT116", "HEK": "HEK293T"
})

intensity = raw.pivot_table(
    index="gene",
    columns="context",
    values="intensity",
    aggfunc="sum",
)
intensity.columns = [f"intensity_{c}" for c in intensity.columns]
intensity["n_protein_ids"] = raw.groupby("gene")["protein_id"].nunique()
quantified = intensity[[
    "intensity_HEK293T",
    "intensity_HCT116",
]].notna().all(axis=1)
# log2FC = log2(HEK293T / HCT116)
intensity["log2FC"] = np.where(
    quantified,
    np.log2(intensity["intensity_HEK293T"]
            / intensity["intensity_HCT116"]),
    np.nan,
)

In [ ]:
# Sort by group, then by absolute log2FC descending
ranking = groups.merge(intensity, left_on="gene", right_index=True, how="left")
ranking["n_protein_ids"] = ranking["n_protein_ids"].astype("Int64")
ranking["abs_log2FC"] = ranking["log2FC"].abs()
ranking["quantified"] = ranking["log2FC"].notna().astype(int)

is_specific = (ranking["group"] == "specific").astype(float)
ranking["score"] = (
    float(np.ceil(ranking["abs_log2FC"].max()) + 1) * is_specific
    + ranking["abs_log2FC"].fillna(-1.0)
)
ranking = (
    ranking.sort_values(["score", "gene"], ascending=[False, True])
    .reset_index(drop=True)
)
ranking["rank"] = np.arange(1, len(ranking) + 1)
ranking = ranking[[
    "gene",
    "score",
    "rank",
    "group",
    "specific_in",
    "mutual_peak",
    "log2FC",
    "abs_log2FC",
    "intensity_HEK293T",
    "intensity_HCT116",
    "n_protein_ids",
    "quantified",
]]
ranking.to_csv(
    "../../result/HEK293T_HCT116/ranking/SEC-MS_abd.ranking",
    sep="\t",
    index=False,
    float_format="%.6g",
)

### Co-elution degree

In [ ]:
# EPIC co-elution network of each line, the one node2vec embeds
networks = {
    "HCT116": pd.read_csv(
        "../../result/HEK293T_HCT116/input/SECMX-HCT116.ppi",
        sep="\t"
    ),
    "HEK293T": pd.read_csv(
        "../../result/HEK293T_HCT116/input/SECMX-HEK293T.ppi",
        sep="\t"
    )
}
n_pairs = {context: len(ppi) for context, ppi in networks.items()}

degree = pd.DataFrame({
    f"n_{context}": pd.concat(
        [ppi["GeneA"], ppi["GeneB"]]
    ).value_counts()
    for context, ppi in networks.items()
}).fillna(0).astype(int)
degree.index.name = "gene"

### Degree-imbalance test

In [ ]:
# Global success probability under the null
p_num = n_pairs["HEK293T"] / sum(n_pairs.values())

# Two-sided binomial on the numerator line's share of partners
k = degree["n_HEK293T"].to_numpy()
n = k + degree["n_HCT116"].to_numpy()
lower = binom.cdf(k, n, p_num)
upper = binom.sf(k - 1, n, p_num)
degree["p_degree"] = np.minimum(1.0, 2.0 * np.minimum(lower, upper))
degree["log2FC"] = np.log2(
    ((degree["n_HEK293T"] + 0.5) / n_pairs["HEK293T"])
    / ((degree["n_HCT116"] + 0.5) / n_pairs["HCT116"])
)
degree["fdr"] = false_discovery_control(
    degree["p_degree"].to_numpy(), method="bh"
)
# make -0.0 as 0.0
degree["score"] = 0.0 - np.log10(degree["p_degree"])

### Degree ranking

In [ ]:
degree_ranking = (
    degree.reset_index()
    .sort_values(
        ["score", "log2FC", "gene"],
        ascending=[False, False, True],
        key=lambda s: s.abs() if s.name == "log2FC" else s,
    )
    .reset_index(drop=True)
)
degree_ranking["rank"] = np.arange(1, len(degree_ranking) + 1)
degree_ranking = degree_ranking[[
    "gene",
    "score",
    "rank",
    "log2FC",
    "n_HEK293T",
    "n_HCT116",
    "p_degree",
    "fdr",
]]
degree_ranking.to_csv(
    "../../result/HEK293T_HCT116/ranking/SEC-MS_degree.ranking",
    sep="\t", index=False
)